# PathGuard: Reproducible Benchmark Analysis

**Purpose:** reproduce descriptive statistics from the benchmark ZIP and, only after independent labels are supplied, calculate exploratory ranking metrics. This notebook does not create or infer ground-truth labels.

**Current collection:** nine pinned public repositories; 8,659 dependency entries; 430 raw PathGuard findings; 427 unique candidate cases. Both scanners found the same union cases in this cohort, so collection counts do not demonstrate a ranking or detection advantage.

**Research integrity:** blank labels stay blank. Do not report precision/recall/nDCG until `reachable`, `not_reachable`, and `unknown` labels are independently reviewed and adjudicated. For publication-grade results, also run PathGuard's canonical TypeScript evaluator and preserve its JSON output.

## Step 1 — Upload the benchmark ZIP
Download `pathguard-expanded-benchmark.zip` from the project handoff, then run the next cell and select that ZIP.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_names, 'Upload the pathguard benchmark ZIP file.'
ZIP_PATH = '/content/' + zip_names[0]
print('Uploaded:', ZIP_PATH)


In [ ]:
import os, zipfile, json, glob
import pandas as pd
from pathlib import Path

ROOT = Path('/content/pathguard_benchmark')
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(ROOT)
candidates_path = ROOT / 'benchmark-candidates.csv'
baseline_path = ROOT / 'osv-scanner-scores.csv'
manifest_path = ROOT / 'experiment-manifest.json'
assert candidates_path.exists() and baseline_path.exists() and manifest_path.exists(), 'ZIP does not contain expected PathGuard benchmark files.'
candidates = pd.read_csv(candidates_path, keep_default_na=False)
baseline = pd.read_csv(baseline_path, keep_default_na=False)
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('Manifest label status:', manifest.get('labelStatus'))
print('Pinned repositories:', len(manifest.get('repositoryCommits', {})))
print('Candidate rows:', len(candidates), '| unique case IDs:', candidates.case_id.nunique())
print('Baseline rows:', len(baseline), '| unique case IDs:', baseline.case_id.nunique())
assert candidates.case_id.is_unique, 'Candidate case IDs must be unique for this analysis.'
assert baseline.case_id.is_unique, 'Baseline case IDs must be unique for this analysis.'
candidates.head(3)


## Step 2 — Descriptive results
This section is safe to use for the current review because it reports collection/coverage facts, not predictive performance.

In [ ]:
import matplotlib.pyplot as plt

repo_summary = (candidates.groupby('repository_url', dropna=False)
    .agg(unique_cases=('case_id','nunique'), packages=('package_name','nunique'))
    .reset_index().sort_values('unique_cases', ascending=False))
repo_summary['repository'] = repo_summary.repository_url.str.replace('https://github.com/','',regex=False)
display(repo_summary[['repository','unique_cases','packages']])

print('\nSource assessment counts:')
display(candidates.source_assessment.value_counts(dropna=False).rename_axis('source_assessment').reset_index(name='cases'))
print('\nDiscovery source counts (multi-source strings are preserved as collected):')
display(candidates.discovery_source.value_counts(dropna=False).rename_axis('discovery_source').reset_index(name='cases'))

for col in ['cvss_score','epss_score','pathguard_score']:
    candidates[col + '_numeric'] = pd.to_numeric(candidates[col], errors='coerce')
coverage = pd.DataFrame({
    'signal':['CVSS','EPSS','PathGuard score','Curated vulnerable-symbol mapping'],
    'available_cases':[
        candidates.cvss_score_numeric.notna().sum(),
        candidates.epss_score_numeric.notna().sum(),
        candidates.pathguard_score_numeric.notna().sum(),
        (~candidates.source_assessment.eq('no-curated-symbol-mapping')).sum()
    ],
    'total_cases':[len(candidates)]*4
})
coverage['coverage_percent'] = (100*coverage.available_cases/coverage.total_cases).round(2)
display(coverage)

plt.figure(figsize=(9,4))
candidates.groupby('repository_url').size().sort_values().plot(kind='barh')
plt.title('Unique candidate cases by repository')
plt.xlabel('Candidate cases')
plt.ylabel('Repository URL')
plt.tight_layout()
plt.show()


## Step 3 — Optional independently reviewed labels
Only run this after the two reviewers have completed the blinded packets and disagreements have been adjudicated. Upload the final merged `labeled-cases.csv` produced by the repository workflow. This cell refuses to compute performance metrics if there are no valid labels. `unknown` cases are reported separately and excluded from the primary binary ranking metrics.

In [ ]:
from google.colab import files
print('Upload final adjudicated labeled-cases.csv now. If labels are not ready, cancel this upload and skip the next cell.')
label_upload = files.upload()
label_files = [name for name in label_upload if name.lower().endswith('.csv') and 'label' in name.lower()]
if label_files:
    labels_df = pd.read_csv('/content/' + label_files[0], keep_default_na=False)
    print('Loaded label file:', label_files[0], 'rows:', len(labels_df))
else:
    labels_df = None
    print('No label CSV uploaded. Performance metrics must not be reported.')


In [ ]:
import numpy as np

def dcg(rels, k):
    rels = np.asarray(rels, dtype=float)[:k]
    if len(rels) == 0: return 0.0
    return float(np.sum((2**rels - 1) / np.log2(np.arange(2, len(rels)+2))))

def ranking_metrics(frame, score_col, k=10):
    ordered = frame.sort_values([score_col, 'case_id'], ascending=[False, True], kind='mergesort')
    rel = ordered['relevant'].astype(int).tolist()
    positives = int(frame.relevant.sum())
    top = rel[:k]
    precision = (sum(top)/k) if len(top) else 0.0
    recall = (sum(top)/positives) if positives else float('nan')
    ideal = sorted(rel, reverse=True)[:k]
    ndcg = (dcg(top,k)/dcg(ideal,k)) if dcg(ideal,k) else float('nan')
    return {'method':score_col, 'n_labeled':len(frame), 'positive_cases':positives,
            f'precision@{k}':precision, f'recall@{k}':recall, f'nDCG@{k}':ndcg}

if labels_df is None:
    print('SKIPPED: no adjudicated label CSV was supplied.')
else:
    required = {'case_id','label'}
    if not required.issubset(labels_df.columns):
        raise ValueError('Label CSV must include case_id and label columns.')
    labels_df = labels_df[['case_id','label']].copy()
    labels_df['label'] = labels_df['label'].astype(str).str.strip().str.lower()
    allowed = {'reachable','not_reachable','unknown'}
    bad = sorted(set(labels_df.label)-allowed)
    if bad: raise ValueError(f'Invalid label values: {bad}')
    if labels_df.case_id.duplicated().any(): raise ValueError('Label CSV has duplicate case IDs.')
    merged = candidates.merge(labels_df, on='case_id', how='left', validate='one_to_one', suffixes=('_candidate',''))
    merged['label'] = merged['label'].fillna('unknown')
    print('Label counts:')
    display(merged.label.value_counts().rename_axis('label').reset_index(name='cases'))
    eval_df = merged[merged.label.isin(['reachable','not_reachable'])].copy()
    if eval_df.empty:
        print('SKIPPED: no reachable/not_reachable adjudicated labels are available.')
    else:
        eval_df['relevant'] = eval_df.label.eq('reachable').astype(int)
        eval_df['cvss_only'] = pd.to_numeric(eval_df.cvss_score, errors='coerce').fillna(0)
        eval_df['epss_only'] = pd.to_numeric(eval_df.epss_score, errors='coerce').fillna(0)
        eval_df['pathguard_v1'] = pd.to_numeric(eval_df.pathguard_score, errors='coerce').fillna(0)
        eval_df = eval_df.merge(baseline[['case_id','score']].rename(columns={'score':'osv_cvss'}), on='case_id', how='left', validate='one_to_one')
        eval_df['osv_cvss'] = pd.to_numeric(eval_df.osv_cvss, errors='coerce').fillna(0)
        K = min(10, len(eval_df))
        metrics = pd.DataFrame([ranking_metrics(eval_df, c, k=K) for c in ['cvss_only','epss_only','pathguard_v1','osv_cvss']])
        display(metrics)
        print('Exploratory only: confirm results with PathGuard canonical evaluator; report label protocol, denominator, confidence intervals, and error analysis.')


## Interpretation checklist for the paper
- Report current figures as **descriptive collection results**, not model performance.
- The candidate set is the union of two scanners; vulnerabilities missed by both tools are not represented.
- CVSS baseline scores with no parseable CVSS use zero in the exported file and are marked by `score_source`.
- Static import/call evidence is not proof of runtime reachability or exploitability.
- Do not claim PathGuard improves ranking until independently labeled cases and predeclared comparisons support that claim.
- Save this notebook and its outputs with the experiment artifact and record the exact artifact, commit, label version, and analysis date.